# A-001 Simple SQL + RAG

**LlamaIndex | PDF only | chunk 400 | overlap 100 | top_k 3**

In [ ]:
%pip install -q llama-index-core==0.14.24 llama-index-readers-file llama-index-embeddings-huggingface sentence-transformers pypdf mlflow

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from mlflow.deployments import get_deploy_client

DOCUMENT_FOLDER = "/Workspace/Nuclear_Enterprise_360/A001 Documents"
ASSET_TABLE = "workspace.nuclear_enterprise_360.asset_360"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"

Settings.chunk_size = 400
Settings.chunk_overlap = 100
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

documents = SimpleDirectoryReader(
    input_dir=DOCUMENT_FOLDER,
    required_exts=[".pdf"]
).load_data()

index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(similarity_top_k=3)

client = get_deploy_client("databricks")

print("PDF documents:", len(documents))
print("Chunk size: 400 | Overlap: 100 | Top K: 3")

In [ ]:
def sql_tool():
    row = spark.sql(f"""
        SELECT asset_id, asset_name, criticality, health_score, risk_level,
               open_work_orders, high_priority_open_work, follow_up_findings
        FROM {ASSET_TABLE}
        WHERE asset_id = 'A-001'
        LIMIT 1
    """).first()
    return row.asDict() if row else {}


def rag_tool(question):
    results = retriever.retrieve(question)
    return "\n\n".join(
        f"SOURCE: {r.node.metadata.get('file_name', 'PDF')}\n{r.node.get_content()}"
        for r in results
    )


def llm(prompt):
    response = client.predict(
        endpoint=CHAT_MODEL,
        inputs={
            "messages": [
                {"role": "system", "content": "Use only the supplied evidence. Do not authorize maintenance or diagnose equipment failure."},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.1,
            "max_tokens": 700
        }
    )
    return response["choices"][0]["message"]["content"]


def ask(question):
    return llm(f"""
QUESTION:
{question}

SQL FACTS:
{sql_tool()}

TOP 3 DOCUMENT CHUNKS:
{rag_tool(question)}

Answer using the SQL facts and document evidence.
APPROVED documents are authoritative.
DRAFT and SUPERSEDED documents are not current authority.
End with the recommended human next step.
""")

In [ ]:
question = "Does A-001 need qualified reliability review, and which approved procedure applies?"
print(ask(question))